# A portfolio analyst agent with Cedar authorization

The agent of [`mongodb_analytics_brave.ipynb`](mongodb_analytics_brave.ipynb)
over `sample_analytics`, with two changes: Brave Search is gone, and **every
tool call is authorized with [Cedar](https://cedarpolicy.com/)** before it
runs. The MongoDB tools are the same aggregation pipelines as before.

Cedar is a policy language from AWS: `permit` and `forbid` statements over a
*principal* (who), an *action* (what) and a *resource*, with conditions on a
*context*. It is default-deny (no matching `permit`, no access) and a `forbid`
always wins over a `permit`. Strands ships a `CedarAuthorization` intervention
that sits at the tool-call boundary and maps each call to a Cedar request,
following the examples in [`../cedar-auth/`](../cedar-auth/):

| Cedar | Here |
|---|---|
| Principal | `User::"<user_id>"`, taken from the agent's `invocation_state` |
| Action | `Action::"<tool name>"`: `find_customers`, `customer_portfolio`, `symbol_activity` |
| Resource | `Resource::"agent"`, fixed |
| `context.input` | the tool's arguments, e.g. `{ name: "Bryce Bowen" }` |
| `context.session` | `hour_utc` and `call_count` (calls to this tool so far in the run) |

Three users with three roles, defined as **entities** rather than in the
prompt, so the model can't talk its way into another role:

| User | Role | Can |
|---|---|---|
| `alice` | manager | everything |
| `bob` | advisor | look customers up, see bank-wide activity, and open the portfolios of **his own book** (Bryce Bowen, Tonya Vega) |
| `carol` | analyst | bank-wide activity only, no customer data |

Plus a `forbid` that caps portfolio lookups at three per run for everyone.
The policy, schema and entities live in [`policies/`](policies/). Needs
`MONGODB_CONNECTION` in `.env` and the `cedar` extra of Strands
(`strands-agents[cedar]`, already in `pyproject.toml`).

In [1]:
import json
import os
import re
from pathlib import Path

import cedarpy
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pymongo import MongoClient
from strands import Agent, tool
from strands.models import BedrockModel
from strands.vended_interventions.cedar import CedarAuthorization

load_dotenv()
YOUR_BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-5-5"

POLICIES_DIR = Path("policies")
POLICY_FILE = POLICIES_DIR / "analytics_access.cedar"
SCHEMA_FILE = POLICIES_DIR / "analytics.cedarschema"
ENTITIES_FILE = POLICIES_DIR / "analytics_entities.json"

mongo_client = MongoClient(os.environ["MONGODB_CONNECTION"])
db = mongo_client["sample_analytics"]
customers, accounts, transactions = db["customers"], db["accounts"], db["transactions"]
for coll in (customers, accounts, transactions):
    print(f"{coll.name}: {coll.estimated_document_count():,} documents")

customers: 500 documents
accounts: 1,746 documents
transactions: 1,746 documents


## 1. The MongoDB tools

Unchanged from `mongodb_analytics_brave.ipynb`, which explains the data and
the pipelines: `find_customers` joins customers to their accounts,
`customer_portfolio` nets a customer's trades per ticker, `symbol_activity`
aggregates the whole bank per ticker.

In [2]:
for coll in (accounts, transactions):
    coll.create_index("account_id", name="account_id_1")  # no-op if it exists

In [3]:
# tier_and_details is a map keyed by an opaque id; this turns it into a plain
# list of tier names so it can be matched and displayed.
TIERS = {"$map": {"input": {"$objectToArray": "$tier_and_details"}, "as": "t", "in": "$$t.v.tier"}}

# One trade per document after $unwind; these sum shares and cash by side.
IS_BUY = {"$eq": ["$tx.transactions.transaction_code", "buy"]}
IS_SELL = {"$eq": ["$tx.transactions.transaction_code", "sell"]}
TOTAL = {"$toDouble": "$tx.transactions.total"}  # stored as a string

JOIN_ACCOUNTS = {"$lookup": {"from": "accounts", "localField": "accounts", "foreignField": "account_id", "as": "accs"}}
JOIN_TRADES = {"$lookup": {"from": "transactions", "localField": "accounts", "foreignField": "account_id", "as": "tx"}}


def money(value):
    return f"${value:,.2f}"


@tool
def find_customers(
    name: str | None = None,
    tier: str | None = None,
    product: str | None = None,
    limit: int = 10,
) -> str:
    """Find customers by name, membership tier and/or account product, with their accounts.

    Args:
        name: Part of the customer's name, case-insensitive, e.g. "Perkins".
        tier: Membership tier: Bronze, Silver, Gold or Platinum.
        product: An account product: InvestmentStock, Brokerage, CurrencyService,
            InvestmentFund, Derivatives or Commodity.
        limit: Maximum number of customers to return.
    """
    pipeline = [{"$addFields": {"tiers": TIERS}}]
    match = {}
    if name:
        match["name"] = re.compile(re.escape(name), re.IGNORECASE)
    if tier:
        match["tiers"] = re.compile(f"^{re.escape(tier)}$", re.IGNORECASE)
    if match:
        pipeline.append({"$match": match})
    pipeline.append(JOIN_ACCOUNTS)
    if product:
        pipeline.append({"$match": {"accs.products": re.compile(f"^{re.escape(product)}$", re.IGNORECASE)}})
    pipeline += [{"$sort": {"name": 1}}, {"$facet": {"total": [{"$count": "n"}], "rows": [{"$limit": limit}]}}]

    result = next(customers.aggregate(pipeline))
    total = result["total"][0]["n"] if result["total"] else 0
    rows = result["rows"]
    print(f"[find_customers] {json.dumps({k: v for k, v in dict(name=name, tier=tier, product=product).items() if v})}: {total} matches")

    if not rows:
        return "No customers match these filters."

    lines = [f"{total} customers match, showing {len(rows)}:"]
    for c in rows:
        benefits = sorted({b for t in c["tier_and_details"].values() for b in t["benefits"]})
        lines.append(
            f"\n{c['name']} ({c['email']}, {'active' if c.get('active') else 'inactive'}) "
            f"— tiers: {', '.join(sorted(set(c['tiers']))) or 'none'}; benefits: {', '.join(benefits) or 'none'}"
        )
        for a in c["accs"]:
            lines.append(f"  account {a['account_id']}: limit {money(a['limit'])}; products: {', '.join(a['products'])}")
    return "\n".join(lines)


@tool
def customer_portfolio(name: str) -> str:
    """Net stock position per ticker for one customer, across all their accounts.

    Nets every buy against every sell. The data is synthetic, so a position can
    be negative (more shares sold than bought); report it as it is.

    Args:
        name: The customer's full name, case-insensitive, e.g. "Bryce Bowen".
    """
    pipeline = [
        {"$match": {"name": re.compile(f"^{re.escape(name)}$", re.IGNORECASE)}},
        JOIN_TRADES,
        {"$unwind": "$tx"},
        {"$unwind": "$tx.transactions"},
        {"$group": {
            "_id": "$tx.transactions.symbol",
            "bought": {"$sum": {"$cond": [IS_BUY, "$tx.transactions.amount", 0]}},
            "sold": {"$sum": {"$cond": [IS_SELL, "$tx.transactions.amount", 0]}},
            "spent": {"$sum": {"$cond": [IS_BUY, TOTAL, 0]}},
            "received": {"$sum": {"$cond": [IS_SELL, TOTAL, 0]}},
            "trades": {"$sum": 1},
            "first": {"$min": "$tx.transactions.date"},
            "last": {"$max": "$tx.transactions.date"},
        }},
        {"$sort": {"_id": 1}},
    ]
    rows = list(customers.aggregate(pipeline))
    print(f"[customer_portfolio] {name!r}: {len(rows)} tickers")

    if not rows:
        if customers.count_documents({"name": re.compile(f"^{re.escape(name)}$", re.IGNORECASE)}) == 0:
            return f"NOT IN DATASET: no customer named {name!r}."
        return f"{name} has no trades."

    lines = [f"Net positions of {name} (shares bought - sold; cash is spent on buys minus received from sells):"]
    for r in rows:
        lines.append(
            f"- {r['_id'].upper()}: net {r['bought'] - r['sold']:,} shares "
            f"(bought {r['bought']:,}, sold {r['sold']:,}) in {r['trades']} trades, "
            f"{r['first']:%Y-%m-%d} to {r['last']:%Y-%m-%d}; net cash {money(r['received'] - r['spent'])}"
        )
    return "\n".join(lines)


@tool
def symbol_activity(symbol: str | None = None, limit: int = 18) -> str:
    """Bank-wide trading activity per ticker: trades, accounts, traded volume, last trade.
    Call it before any web search about a ticker: one it doesn't list is not in the dataset.

    Args:
        symbol: One ticker, e.g. "AAPL", for a single row. Omit for every ticker.
        limit: Maximum number of tickers to return, largest volume first.
    """
    pipeline = [{"$unwind": "$transactions"}]
    if symbol:
        pipeline.append({"$match": {"transactions.symbol": symbol.lower()}})
    pipeline += [
        {"$group": {
            "_id": "$transactions.symbol",
            "trades": {"$sum": 1},
            "accounts": {"$addToSet": "$account_id"},
            "volume": {"$sum": {"$toDouble": "$transactions.total"}},
            # $top picks the latest trade per group without sorting the 88,000
            # unwound trades first, which exceeds the 32 MB limit of a $sort.
            "last": {"$top": {
                "sortBy": {"transactions.date": -1},
                "output": {"date": "$transactions.date", "price": {"$toDouble": "$transactions.price"}},
            }},
        }},
        {"$project": {"trades": 1, "volume": 1, "last": 1, "accounts": {"$size": "$accounts"}}},
        {"$sort": {"volume": -1}},
        {"$limit": limit},
    ]
    rows = list(transactions.aggregate(pipeline))
    print(f"[symbol_activity] {symbol or 'all'}: {len(rows)} tickers")

    if not rows:
        return f"NOT IN DATASET: no trades for ticker {symbol!r}."
    return "\n".join(
        f"- {r['_id'].upper()}: {r['trades']:,} trades by {r['accounts']} accounts, "
        f"volume {money(r['volume'])}, last trade {r['last']['date']:%Y-%m-%d} at {money(r['last']['price'])}"
        for r in rows
    )

## 2. The policies

Three files in `policies/`:

- `analytics_access.cedar`: the rules. Four `permit`s, one per role (the
  advisor has two), and one `forbid`.
- `analytics.cedarschema`: the types. It declares the three actions with the
  shape of their `context.input` (mirroring each tool's arguments) and the
  `User`/`Role`/`Resource` entities. With a schema, Cedar validates the
  policies when they load (a typo in a tool name fails right away) and every
  request when it is evaluated.
- `analytics_entities.json`: the data. Users, their roles (as parents in
  the entity hierarchy, which is what `principal in Role::"advisor"`
  checks) and Bob's `book` attribute.

In [4]:
print(POLICY_FILE.read_text())

// Who can call which tool of the portfolio analyst agent.
//
// Cedar is default-deny: a tool call is allowed only if a permit matches
// and no forbid does. Roles come from the entity hierarchy in
// analytics_entities.json (User::"alice" in Role::"manager"), not from the
// prompt, so the model cannot talk its way into another role.

// Managers can use every tool.
@id("manager-full-access")
permit (
  principal in Role::"manager",
  action,
  resource
);

// Advisors can look customers up and see bank-wide activity...
@id("advisor-lookups")
permit (
  principal in Role::"advisor",
  action in [Action::"find_customers", Action::"symbol_activity"],
  resource
);

// ...but may only open the portfolios of the customers in their book.
// The book is an attribute of the User entity; the name is a tool argument.
@id("advisor-own-book")
permit (
  principal in Role::"advisor",
  action == Action::"customer_portfolio",
  resource
)
when {
  principal has book && principal.book.contains(con

In [5]:
entities = json.loads(ENTITIES_FILE.read_text())
for e in entities:
    if e["uid"]["type"] == "User":
        roles = ", ".join(p["id"] for p in e["parents"])
        print(f'User::"{e["uid"]["id"]}" in Role::"{roles}"  attrs={e["attrs"]}')

User::"alice" in Role::"manager"  attrs={}
User::"bob" in Role::"advisor"  attrs={'book': ['Bryce Bowen', 'Tonya Vega']}
User::"carol" in Role::"analyst"  attrs={}


### Decisions without an agent

`cedarpy.is_authorized` is what the handler calls. Building the same request
by hand shows how each rule plays out, with no model involved. Cedar names
the policies `policy0`, `policy1`, ... in file order in its diagnostics, so
`policy4` is the `forbid`.

In [6]:
policies = POLICY_FILE.read_text()
schema = SCHEMA_FILE.read_text()


def decide(user_id, tool_name, tool_input, call_count=1):
    """Same request shape as CedarAuthorization.before_tool_call builds."""
    request = {
        "principal": f'User::"{user_id}"',
        "action": f'Action::"{tool_name}"',
        "resource": 'Resource::"agent"',
        "context": {
            "input": tool_input,
            "session": {"hour_utc": 14, "call_count": call_count},
        },
    }
    result = cedarpy.is_authorized(request, policies, entities, schema)
    return result.decision.name, ", ".join(result.diagnostics.reasons) or "no policy matched"


cases = [
    ("alice", "customer_portfolio", {"name": "Elizabeth Ray"}, 1),
    ("alice", "customer_portfolio", {"name": "Elizabeth Ray"}, 4),
    ("bob", "customer_portfolio", {"name": "Bryce Bowen"}, 1),
    ("bob", "customer_portfolio", {"name": "Elizabeth Ray"}, 1),
    ("bob", "find_customers", {"tier": "Platinum"}, 1),
    ("carol", "symbol_activity", {"limit": 5}, 1),
    ("carol", "find_customers", {"tier": "Platinum"}, 1),
    ("mallory", "symbol_activity", {}, 1),
]
rows = ["| user | tool | input | call # | decision | matched |", "|---|---|---|---|---|---|"]
for user_id, tool_name, tool_input, n in cases:
    decision, matched = decide(user_id, tool_name, tool_input, n)
    rows.append(f"| {user_id} | `{tool_name}` | `{json.dumps(tool_input)}` | {n} | **{decision}** | {matched} |")
display(Markdown("\n".join(rows)))

| user | tool | input | call # | decision | matched |
|---|---|---|---|---|---|
| alice | `customer_portfolio` | `{"name": "Elizabeth Ray"}` | 1 | **Allow** | policy0 |
| alice | `customer_portfolio` | `{"name": "Elizabeth Ray"}` | 4 | **Deny** | policy4 |
| bob | `customer_portfolio` | `{"name": "Bryce Bowen"}` | 1 | **Allow** | policy2 |
| bob | `customer_portfolio` | `{"name": "Elizabeth Ray"}` | 1 | **Deny** | no policy matched |
| bob | `find_customers` | `{"tier": "Platinum"}` | 1 | **Allow** | policy1 |
| carol | `symbol_activity` | `{"limit": 5}` | 1 | **Allow** | policy3 |
| carol | `find_customers` | `{"tier": "Platinum"}` | 1 | **Deny** | no policy matched |
| mallory | `symbol_activity` | `{}` | 1 | **Deny** | no policy matched |

## 3. The agent

`CedarAuthorization` loads the three files and validates the policy against
the schema. It keeps the per-tool `call_count` in the agent's state, so each
agent gets its own instance. The principal comes from `invocation_state`,
which the caller passes with the question; no `user_id` there means no
principal, and every call is denied.

A denied call never runs. The model gets the denial as the tool result and
is told to report it, not to look for another way to the data.

In [7]:
class LoggingCedar(CedarAuthorization):
    """CedarAuthorization that prints every decision, so denials show in the notebook."""

    def before_tool_call(self, event, **kwargs):
        decision = super().before_tool_call(event, **kwargs)
        user_id = event.invocation_state.get("user_id", "<nobody>")
        print(
            f"[cedar] {user_id}: {event.tool_use['name']}({json.dumps(event.tool_use.get('input') or {})}) "
            f"-> {type(decision).__name__}{': ' + decision.reason if getattr(decision, 'reason', None) else ''}"
        )
        return decision


def cedar_for_agent():
    return LoggingCedar(
        policies=str(POLICY_FILE),
        schema=str(SCHEMA_FILE),
        entities=str(ENTITIES_FILE),
        principal_resolver=lambda state: (
            {"type": "User", "id": state["user_id"]} if state.get("user_id") else None
        ),
    )


SYSTEM_PROMPT = """You are an analyst for a bank's customer database in MongoDB (customers, accounts and stock trades up to 2017).

## Rules

1. Answer only from the tools. If a tool call is denied by the authorization policy, tell the user which action was denied and the reason given, and do not try other tools or arguments to get the same data another way.
2. Answer the parts of the question you are authorized to answer, and say plainly which parts you could not.
3. The trades are synthetic and end in 2017: positions can be negative. Report what the data says.
4. Answer in the language of the question. Keep answers concise.
"""

model = BedrockModel(
    model_id=YOUR_BEDROCK_MODEL_ID,
    region_name=os.getenv("AWS_REGION", "us-west-2"),
    max_tokens=4096,
)


def ask(question, user_id=None):
    # A fresh agent and handler per question: call counts start at zero
    agent = Agent(
        name="Portfolio Analyst",
        model=model,
        system_prompt=SYSTEM_PROMPT,
        tools=[find_customers, customer_portfolio, symbol_activity],
        interventions=[cedar_for_agent()],
        callback_handler=None,  # don't also stream the answer to stdout
    )

    display(Markdown(f"**User:** `{user_id or 'none'}`"))
    response = agent(question, invocation_state={"user_id": user_id} if user_id else {})

    # Tools the agent actually called (absent from the dict means never called)
    used = response.metrics.tool_metrics
    usage = response.metrics.accumulated_usage
    display(Markdown(
        "**Tools used:** "
        + (", ".join(f"`{name}` x{m.call_count}" for name, m in used.items()) or "_none_")
        + f" — {usage['inputTokens']:,} input / {usage['outputTokens']:,} output tokens"
    ))

    display(Markdown(str(response)))

Alice is a manager: both calls go through.

In [8]:
ask("¿Qué cuentas tiene Elizabeth Ray y cuál es su cartera de acciones?", user_id="alice")

**User:** `alice`

[cedar] alice: find_customers({"name": "Elizabeth Ray"}) -> Proceed
[cedar] alice: customer_portfolio({"name": "Elizabeth Ray"}) -> Proceed
[find_customers] {"name": "Elizabeth Ray"}: 1 matches
[customer_portfolio] 'Elizabeth Ray': 5 tickers


**Tools used:** `find_customers` x1, `customer_portfolio` x1 — 3,174 input / 775 output tokens

**Elizabeth Ray** es cliente activa de nivel Bronze (arroyocolton@gmail.com). Sus beneficios son línea dedicada 24 h, conserjería y entradas deportivas.

**Cuentas**

| Cuenta | Límite | Productos |
|---|---|---|
| 276528 | $10,000 | InvestmentFund, InvestmentStock |
| 324287 | $10,000 | Commodity, CurrencyService, Derivatives, InvestmentStock |
| 332179 | $10,000 | Commodity, CurrencyService, InvestmentFund, Brokerage, InvestmentStock |
| 371138 | $9,000 | Derivatives, InvestmentStock |
| 387979 | $10,000 | Brokerage, Derivatives, InvestmentFund, Commodity, InvestmentStock |
| 422649 | $10,000 | CurrencyService, InvestmentStock |

**Cartera de acciones (posición neta en todas sus cuentas)**

| Ticker | Posición neta | Comprado | Vendido | Operaciones | Efectivo neto |
|---|---|---|---|---|---|
| AMD | 32,198 | 188,017 | 155,819 | 66 | $16,521.06 |
| CRM | -4,390 | 189,031 | 193,421 | 79 | $186,904.20 |
| CSCO | -29,437 | 136,123 | 165,560 | 62 | $688,125.91 |
| INTC | 33,198 | 193,390 | 160,192 | 74 | -$542,047.70 |
| NVDA | -22,450 | 148,196 | 170,646 | 69 | $834,548.31 |

Los datos son sintéticos y terminan en 2017. Por eso hay posiciones netas negativas (CRM, CSCO y NVDA), donde vendió más acciones de las que compró. Las muestro tal como aparecen en los datos.


Bob is an advisor. Bryce Bowen is in his book, Elizabeth Ray isn't: same tool,
two decisions, decided by `principal.book.contains(context.input.name)`.

In [9]:
ask("Show me the stock portfolios of Bryce Bowen and Elizabeth Ray.", user_id="bob")

**User:** `bob`

[cedar] bob: customer_portfolio({"name": "Bryce Bowen"}) -> Proceed
[cedar] bob: customer_portfolio({"name": "Elizabeth Ray"}) -> Deny: Access denied by Cedar policy
[customer_portfolio] 'Bryce Bowen': 3 tickers


**Tools used:** `customer_portfolio` x2 — 2,747 input / 461 output tokens

**Bryce Bowen** (net positions, shares bought minus sold):

| Ticker | Bought | Sold | Net shares | Trades | Period | Net cash |
|---|---|---|---|---|---|---|
| CSCO | 71,938 | 85,929 | −13,991 | 36 | 1992-06-15 to 2016-05-24 | $562,459.17 |
| EBAY | 52,335 | 112,444 | −60,109 | 26 | 2002-03-07 to 2016-10-28 | $915,220.90 |
| INTC | 81,947 | 87,061 | −5,114 | 36 | 1975-01-22 to 2016-05-25 | $303,341.01 |

All three positions are negative. He sold more shares than he bought, which can happen because the data is synthetic.

**Elizabeth Ray:** I couldn't retrieve this portfolio. The call was denied: "Access denied by Cedar policy". The policy gave no further reason. I didn't try other tools or arguments to get the data another way.


Carol is an analyst: aggregates yes, customers no.

In [10]:
ask("¿Qué acción mueve más volumen en el banco? ¿Y qué clientes Platinum tienen el producto Derivatives?", user_id="carol")

**User:** `carol`

[cedar] carol: symbol_activity({"limit": 1}) -> Proceed
[cedar] carol: find_customers({"tier": "Platinum", "product": "Derivatives"}) -> Deny: Access denied by Cedar policy
[symbol_activity] all: 1 tickers


**Tools used:** `find_customers` x1, `symbol_activity` x1 — 2,609 input / 324 output tokens

**Acción con más volumen:** GOOG. Tiene 5.500 operaciones en 562 cuentas, con un volumen de $18.110.672.711,55. La última operación fue el 2017-01-09 a $807,68.

**Clientes Platinum con Derivatives:** no puedo responder esta parte. La búsqueda de clientes por nivel y producto fue denegada por la política de autorización (Cedar policy), que dio el mensaje "Access denied by Cedar policy" sin más detalle. No probé otras herramientas ni otros parámetros para conseguir estos datos.


No `user_id` in `invocation_state`: the resolver returns no principal and the
handler denies before Cedar is even consulted.

In [11]:
ask("Which ticker has the largest traded volume?")

**User:** `none`

[cedar] <nobody>: symbol_activity({"limit": 1}) -> Deny: No principal identity found in invocation state


**Tools used:** `symbol_activity` x1 — 2,374 input / 184 output tokens

The tool call was denied, so I can't tell you which ticker has the largest traded volume. The `symbol_activity` lookup, which returns the bank-wide trading volume per ticker, came back with "DENIED: No principal identity found in invocation state." The authorization policy gave that reason: the request had no principal identity attached.

Per the rules, I haven't tried other tools or arguments to get the same data another way.


The `forbid` applies to everyone. Alice asks for four portfolios: the fourth
call to `customer_portfolio` in the run has `call_count == 4` and is denied,
however much her role permits. Strands runs the tool calls of a turn
concurrently, so which customer gets the denial varies.

In [12]:
ask("Show the stock portfolios of Elizabeth Ray, Bryce Bowen, Tonya Vega and Anna Perkins.", user_id="alice")

**User:** `alice`

[cedar] alice: customer_portfolio({"name": "Elizabeth Ray"}) -> Proceed
[cedar] alice: customer_portfolio({"name": "Bryce Bowen"}) -> Proceed
[cedar] alice: customer_portfolio({"name": "Tonya Vega"}) -> Proceed
[cedar] alice: customer_portfolio({"name": "Anna Perkins"}) -> Deny: Access denied by Cedar policy: policy4
[customer_portfolio] 'Elizabeth Ray': 5 tickers
[customer_portfolio] 'Bryce Bowen': 3 tickers
[customer_portfolio] 'Tonya Vega': 3 tickers


**Tools used:** `customer_portfolio` x4 — 3,528 input / 895 output tokens

Here are the net stock positions for three of the four customers. The data is synthetic, so some positions are negative.

**Elizabeth Ray**
| Ticker | Net shares | Bought | Sold | Trades | Net cash |
|---|---|---|---|---|---|
| AMD | 32,198 | 188,017 | 155,819 | 66 | $16,521.06 |
| CRM | -4,390 | 189,031 | 193,421 | 79 | $186,904.20 |
| CSCO | -29,437 | 136,123 | 165,560 | 62 | $688,125.91 |
| INTC | 33,198 | 193,390 | 160,192 | 74 | -$542,047.70 |
| NVDA | -22,450 | 148,196 | 170,646 | 69 | $834,548.31 |

**Bryce Bowen**
| Ticker | Net shares | Bought | Sold | Trades | Net cash |
|---|---|---|---|---|---|
| CSCO | -13,991 | 71,938 | 85,929 | 36 | $562,459.17 |
| EBAY | -60,109 | 52,335 | 112,444 | 26 | $915,220.90 |
| INTC | -5,114 | 81,947 | 87,061 | 36 | $303,341.01 |

**Tonya Vega**
| Ticker | Net shares | Bought | Sold | Trades | Net cash |
|---|---|---|---|---|---|
| AMZN | 76,054 | 101,120 | 25,066 | 27 | -$14,333,936.82 |
| CRM | -24,338 | 69,948 | 94,286 | 32 | $1,075,662.03 |
| SAP | -20,640 | 78,043 | 98,683 | 33 | $1,730,920.58 |

**Anna Perkins:** I couldn't retrieve this portfolio. The authorization policy denied the request (Cedar policy: policy4). I didn't try other tools or arguments to get her data another way.


## Clean up

The notebook only reads the sample collections.

In [13]:
mongo_client.close()